In [ ]:
import requests

session = requests.Session()

session.post('https://www.space-track.org/ajaxauth/login', data={
    'identity': 'dejuakim@gmail.com',
    'password': 'siawevengersaiffel'
})

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list = response.json()

print(f"수신된 위성 수: {len(tle_list)}")
print("\n위성 이름 샘플 (처음 30개):")
for sat in tle_list[:30]:
    print(f"  {sat['OBJECT_NAME']:<30} 고도추정: MEAN_MOTION={sat['MEAN_MOTION']}")

수신된 위성 수: 13593

위성 이름 샘플 (처음 30개):
  3CAT-4                         고도추정: MEAN_MOTION=15.02085848
  3CAT-5/A (TYVAK-0161)          고도추정: MEAN_MOTION=15.45143577
  3CAT-5/B (TYVAK-0162)          고도추정: MEAN_MOTION=15.46437482
  6GSTARLAB                      고도추정: MEAN_MOTION=15.18217110
  A-SEANSAT-PG1                  고도추정: MEAN_MOTION=15.35876320
  AAC-AIS-SAT-1                  고도추정: MEAN_MOTION=14.91834702
  AAC-AIS-SAT2                   고도추정: MEAN_MOTION=14.85305100
  AAC-AIS-SAT3                   고도추정: MEAN_MOTION=14.84555624
  AAC-HSI-SAT1                   고도추정: MEAN_MOTION=15.12409015
  AAC-HSI-SAT2                   고도추정: MEAN_MOTION=15.20163723
  AAC-HSI-SAT3                   고도추정: MEAN_MOTION=15.15616608
  AAC-IO1                        고도추정: MEAN_MOTION=14.91379200
  AAU CUBESAT                    고도추정: MEAN_MOTION=14.23945998
  AAUSAT3                        고도추정: MEAN_MOTION=14.39899529
  AC1-001                        고도추정: MEAN_MOTION=15.18205753
  AC1-002          

In [2]:
# OBJECT_NAME 기반이 아닌
# RCS_SIZE 기준 추가 (위성 크기)
# 지구관측 위성은 보통 MEDIUM 또는 LARGE

url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/ECCENTRICITY/<0.25"
    "/INCLINATION/>40"
    "/RCS_SIZE/MEDIUM,LARGE"   # 추가
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url)
tle_list_filtered = response.json()
print(f"RCS 필터 후 위성 수: {len(tle_list_filtered)}")

RCS 필터 후 위성 수: 10378


In [3]:
import requests

response = requests.get(
    "https://celestrak.org/SOCRATES/query.php",
    params={"GROUP": "earth-obs", "FORMAT": "tle"}
)
print(f"상태코드: {response.status_code}")
print(response.text[:500])

상태코드: 404
<!DOCTYPE html>
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=iso-8859-1"/>
<title>404 - File Not Found.</title>
<style type="text/css">
body{margin:0;font-size:.7em;font-family:Verdana, Arial, Helvetica, sans-serif;background:#EEEEEE;}
fieldset{padding:0 15px 10px 15px;}
h1{font-size:2.4em;margin:0;color:#FFF;}
h2{font-size:1.7em;margin:0;color:#CC0000;}
h3{font-size:1.2em;margin:10px 0 0 0;color:#000000;}
#header{width:96%;margin:0 0 0 0;padding:6px 2% 6p


In [4]:
import requests

# CelesTrak 새 URL 형식
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/pub/TLE/catalog.txt",
    "https://celestrak.org/SOCRATES/",
    "https://celestrak.org/pub/TLE/eo-ops.txt",
    "https://celestrak.org/pub/TLE/resource.txt",
]

for url in urls_to_try:
    r = requests.get(url, timeout=10)
    print(f"{r.status_code} | {url}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=tle
403 | https://celestrak.org/pub/TLE/catalog.txt
200 | https://celestrak.org/SOCRATES/
403 | https://celestrak.org/pub/TLE/eo-ops.txt
403 | https://celestrak.org/pub/TLE/resource.txt


In [5]:
# CelesTrak GP 데이터 API (새 형식)
urls_to_try = [
    "https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json",
    "https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle",
    "https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle",  # ISS 테스트
    "https://celestrak.org/pub/TLE/active.txt",
    "https://celestrak.org/satcat/records.csv",
]

for url in urls_to_try:
    try:
        r = requests.get(url, timeout=10)
        print(f"{r.status_code} | {url}")
    except Exception as e:
        print(f"ERROR | {url} | {e}")

404 | https://celestrak.org/SOCRATES/query.php?GROUP=earth-obs&FORMAT=json
404 | https://celestrak.org/cgi-bin/TLE.cgi?GROUP=earth-obs&FORMAT=tle
404 | https://celestrak.org/SOCRATES/query.php?CATNR=25544&FORMAT=tle
ERROR | https://celestrak.org/pub/TLE/active.txt | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /pub/TLE/active.txt (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276eecad0>, 'Connection to celestrak.org timed out. (connect timeout=10)'))
ERROR | https://celestrak.org/satcat/records.csv | HTTPSConnectionPool(host='celestrak.org', port=443): Max retries exceeded with url: /satcat/records.csv (Caused by ConnectTimeoutError(<HTTPSConnection(host='celestrak.org', port=443) at 0x24276ef8d90>, 'Connection to celestrak.org timed out. (connect timeout=10)'))


In [2]:
import pandas as pd
df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")
print(df.columns.tolist())
print(f"전체 위성 수: {len(df)}")

['Name of Satellite, Alternate Names', 'Current Official Name of Satellite', 'Country/Org of UN Registry', 'Country of Operator/Owner', 'Operator/Owner', 'Users', 'Purpose', 'Detailed Purpose', 'Class of Orbit', 'Type of Orbit', 'Longitude of GEO (degrees)', 'Perigee (km)', 'Apogee (km)', 'Eccentricity', 'Inclination (degrees)', 'Period (minutes)', 'Launch Mass (kg.)', 'Dry Mass (kg.)', 'Power (watts)', 'Date of Launch', 'Expected Lifetime (yrs.)', 'Contractor', 'Country of Contractor', 'Launch Site', 'Launch Vehicle', 'COSPAR Number', 'NORAD Number', 'Comments', 'Unnamed: 28', 'Source Used for Orbital Data', 'Source', 'Source.1', 'Source.2', 'Source.3', 'Source.4', 'Source.5', 'Source.6', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', '

In [5]:
import pandas as pd

df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")

# EO + SAR 위성 필터링
eo_sar_df = df[
    df['Detailed Purpose'].str.contains(
        'Optical|Multispectral|Hyperspectral|Video|Imaging|Radar|SAR',
        na=False
    )
][['Current Official Name of Satellite', 'NORAD Number', 'Detailed Purpose']].dropna()

eo_sar_df['NORAD Number'] = eo_sar_df['NORAD Number'].astype(int)

print(f"EO+SAR 위성 수: {len(eo_sar_df)}")
print(eo_sar_df['Detailed Purpose'].value_counts())

norad_ids = eo_sar_df['NORAD Number'].tolist()
print(f"\nNORAD ID 수: {len(norad_ids)}")

EO+SAR 위성 수: 735
Detailed Purpose
Optical Imaging                                          516
Radar Imaging                                             98
Multispectral Imaging                                     25
Hyperspectral Imaging                                     23
Infrared Imaging                                          14
Radar Imaging (SAR)                                       12
Radar Imaging/Earth Science                                8
Video Imaging                                              7
Imaging                                                    5
Optical Imaging/Automatic Identification System (AIS)      3
Optical Imaging/Meterology                                 3
Radar Surveillance                                         2
Synthetic Aperture Radar (SAR)                             2
Synthetic Aperture Imaging                                 2
Optical Imaging (video)                                    2
Optical Imaging/Meteorology                        

In [ ]:
import requests
import json
import time

# Space-Track 로그인
USERNAME = "dejuakim@gmail.com"
PASSWORD = "siawevengersaiffel"

login_url = "https://www.space-track.org/ajaxauth/login"
session = requests.Session()
session.post(login_url, data={"identity": USERNAME, "password": PASSWORD})

# NORAD ID 배치 처리 (100개씩)
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    return response.json()

# 100개씩 나눠서 수집
batch_size = 100
all_tle = []

for i in range(0, len(norad_ids), batch_size):
    batch = norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    all_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(norad_ids)}")
    time.sleep(1)  # 요청 간격

print(f"\n총 TLE 수집: {len(all_tle)}개")

# 저장
import os
os.makedirs("project/01_data/raw", exist_ok=True)
with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: ../project/01_data/raw/tle_eo_sar.json")

수집 완료: 100/735
수집 완료: 200/735
수집 완료: 300/735
수집 완료: 400/735
수집 완료: 500/735
수집 완료: 600/735
수집 완료: 700/735
수집 완료: 735/735

총 TLE 수집: 693개
저장 완료: project/01_data/raw/tle_eo_sar.json


In [9]:
# 단일 NORAD ID로 테스트
test_id = norad_ids[0]
print(f"테스트 NORAD ID: {test_id}")

url = (
    "https://www.space-track.org/basicspacedata/query"
    f"/class/gp/NORAD_CAT_ID/{test_id}"
    "/orderby/NORAD_CAT_ID"
    "/format/json"
)

response = session.get(url)
print(f"상태 코드: {response.status_code}")
print(response.text[:300])

테스트 NORAD ID: 44859
상태 코드: 200
[{"CCSDS_OMM_VERS":"3.0","COMMENT":"GENERATED VIA SPACE-TRACK.ORG API","CREATION_DATE":"2026-05-20T02:32:39","ORIGINATOR":"18 SPCS","OBJECT_NAME":"IHOPSAT-TD","OBJECT_ID":"2019-089H","CENTER_NAME":"EARTH","REF_FRAME":"TEME","TIME_SYSTEM":"UTC","MEAN_ELEMENT_THEORY":"SGP4","EPOCH":"2026-05-19T11:37:1


In [11]:
import requests
import json
import time

# 배치 처리 함수 수정
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    
    # 오류 체크
    if response.status_code != 200:
        print(f"오류 발생: {response.status_code}")
        return []
    
    result = response.json()
    
    # "error" 문자열 필터링
    if isinstance(result, list):
        result = [r for r in result if isinstance(r, dict)]
    
    return result

# 100개씩 나눠서 수집
batch_size = 100
all_tle = []

for i in range(0, len(norad_ids), batch_size):
    batch = norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    all_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(norad_ids)} | 현재 수집: {len(all_tle)}개")
    time.sleep(2)

print(f"\n총 TLE 수집: {len(all_tle)}개")

# 저장
import os
os.makedirs("../project/01_data/raw", exist_ok=True)
with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: ../project/01_data/raw/tle_eo_sar.json")

수집 완료: 100/735 | 현재 수집: 98개
수집 완료: 200/735 | 현재 수집: 198개
수집 완료: 300/735 | 현재 수집: 298개
수집 완료: 400/735 | 현재 수집: 384개
수집 완료: 500/735 | 현재 수집: 475개
수집 완료: 600/735 | 현재 수집: 559개
수집 완료: 700/735 | 현재 수집: 658개
수집 완료: 735/735 | 현재 수집: 693개

총 TLE 수집: 693개
저장 완료: ../project/01_data/raw/tle_eo_sar.json


In [1]:
import pandas as pd

df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")

print("Users 종류:")
print(df['Users'].value_counts())
print("\nPurpose 종류:")
print(df['Purpose'].value_counts())

Users 종류:
Users
Commercial                        6080
Government                         558
Military                           457
Civil                              160
Government/Commercial               97
Military/Commercial                 82
Military/Government                 56
Government/Civil                    40
Military/Civil                       7
Government/Military                  4
Commercial/Civil                     4
Civil/Government                     4
Civil/Military                       3
Commercial/Military                  2
Civil/Commercial                     1
Government/Commercial/Military       1
Commercial/Government                1
Commercial                           1
Government                           1
Military                             1
Name: count, dtype: int64

Purpose 종류:
Purpose
Communications                                    5514
Earth Observation                                 1235
Technology Development                         

In [2]:
eo_sar_df = df[
    # 1. 지구관측 목적
    (df['Purpose'].str.contains('Earth Observation', na=False)) &
    # 2. EO 또는 SAR 센서
    (df['Detailed Purpose'].str.contains(
        'Optical|Multispectral|Hyperspectral|Video|Imaging|Radar|SAR',
        na=False
    )) &
    # 3. Commercial 포함된 경우만
    (df['Users'].str.contains('Commercial', na=False))
][['Current Official Name of Satellite', 'NORAD Number', 'Detailed Purpose', 'Users']].dropna()

In [3]:
import pandas as pd

df = pd.read_excel("UCS-Satellite-Database 5-1-2023.xlsx")

eo_sar_df = df[
    (df['Purpose'].str.contains('Earth Observation', na=False)) &
    (df['Detailed Purpose'].str.contains(
        'Optical|Multispectral|Hyperspectral|Video|Imaging|Radar|SAR',
        na=False
    )) &
    (df['Users'].str.contains('Commercial', na=False))
][['Current Official Name of Satellite', 'NORAD Number', 'Detailed Purpose', 'Users']].dropna()

eo_sar_df['NORAD Number'] = eo_sar_df['NORAD Number'].astype(int)

print(f"필터링 결과: {len(eo_sar_df)}개")
print(f"\nDetailed Purpose 분포:")
print(eo_sar_df['Detailed Purpose'].value_counts())
print(f"\nUsers 분포:")
print(eo_sar_df['Users'].value_counts())

norad_ids = eo_sar_df['NORAD Number'].tolist()
print(f"\nNORAD ID 수: {len(norad_ids)}")

필터링 결과: 419개

Detailed Purpose 분포:
Detailed Purpose
Optical Imaging                                          326
Radar Imaging                                             38
Hyperspectral Imaging                                     13
Multispectral Imaging                                     13
Radar Imaging (SAR)                                       12
Video Imaging                                              7
Synthetic Aperture Imaging                                 2
Optical Imaging (video)                                    2
Infrared Imaging                                           1
Subsurface Imaging                                         1
Optical/Infrared Imaging                                   1
Optical/Hyperspectral Imaging                              1
Optical Imaging/Automatic Identification System (AIS)      1
Optical Imaging/Infrared Imaging                           1
Name: count, dtype: int64

Users 분포:
Users
Commercial               408
Government/Commercial 

In [5]:
# EO/SAR 구분 컬럼 추가
def get_sensor_type(purpose):
    if pd.isna(purpose):
        return 'Unknown'
    if 'Radar' in purpose or 'SAR' in purpose or 'Synthetic Aperture' in purpose:
        return 'SAR'
    return 'EO'

eo_sar_df['sensor_type'] = eo_sar_df['Detailed Purpose'].apply(get_sensor_type)

print(eo_sar_df['sensor_type'].value_counts())

# 저장
eo_sar_df.to_csv("../project/01_data/raw/ucs_eo_sar_filtered.csv", index=False)
print("저장 완료: ucs_eo_sar_filtered.csv")

sensor_type
EO     367
SAR     52
Name: count, dtype: int64
저장 완료: ucs_eo_sar_filtered.csv


In [6]:
import requests
import json
import time

# Space-Track 로그인
USERNAME = "dejuakim@gmail.com"
PASSWORD = "siawevengersaiffel"

login_url = "https://www.space-track.org/ajaxauth/login"
session = requests.Session()
session.post(login_url, data={"identity": USERNAME, "password": PASSWORD})

# UCS 필터링된 NORAD ID 로드
import pandas as pd
ucs_df = pd.read_csv("../project/01_data/raw/ucs_eo_sar_filtered.csv")
norad_ids = ucs_df['NORAD Number'].tolist()
print(f"대상 NORAD ID: {len(norad_ids)}개")

# 100개씩 배치 수집
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    if response.status_code != 200:
        return []
    result = response.json()
    if isinstance(result, list):
        result = [r for r in result if isinstance(r, dict)]
    return result

batch_size = 100
all_tle = []

for i in range(0, len(norad_ids), batch_size):
    batch = norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    all_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(norad_ids)} | 현재: {len(all_tle)}개")
    time.sleep(2)

print(f"\n총 TLE 수집: {len(all_tle)}개")

with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: tle_eo_sar.json")

대상 NORAD ID: 419개
수집 완료: 100/419 | 현재: 100개
수집 완료: 200/419 | 현재: 200개
수집 완료: 300/419 | 현재: 300개
수집 완료: 400/419 | 현재: 400개
수집 완료: 419/419 | 현재: 419개

총 TLE 수집: 419개
저장 완료: tle_eo_sar.json


In [7]:
import json

with open("../project/01_data/raw/tle_eo_sar.json") as f:
    tle_data = json.load(f)

print(f"총 위성 수: {len(tle_data)}")
print("\n컬럼 목록:")
for key, value in tle_data[0].items():
    print(f"  {key}: {value}")

총 위성 수: 419

컬럼 목록:
  CCSDS_OMM_VERS: 3.0
  COMMENT: GENERATED VIA SPACE-TRACK.ORG API
  CREATION_DATE: 2026-05-20T17:35:22
  ORIGINATOR: 18 SPCS
  OBJECT_NAME: BEIJING 1 (TSINGHUA)
  OBJECT_ID: 2005-043A
  CENTER_NAME: EARTH
  REF_FRAME: TEME
  TIME_SYSTEM: UTC
  MEAN_ELEMENT_THEORY: SGP4
  EPOCH: 2026-05-20T15:59:36.646368
  MEAN_MOTION: 14.63514002
  ECCENTRICITY: 0.00149089
  INCLINATION: 98.2880
  RA_OF_ASC_NODE: 273.7317
  ARG_OF_PERICENTER: 13.1098
  MEAN_ANOMALY: 347.0488
  EPHEMERIS_TYPE: 0
  CLASSIFICATION_TYPE: U
  NORAD_CAT_ID: 28890
  ELEMENT_SET_NO: 999
  REV_AT_EPOCH: 9662
  BSTAR: 0.00005103871600
  MEAN_MOTION_DOT: 0.00000216
  MEAN_MOTION_DDOT: 0.0000000000000
  SEMIMAJOR_AXIS: 7059.987
  PERIOD: 98.393
  APOAPSIS: 692.378
  PERIAPSIS: 671.327
  OBJECT_TYPE: PAYLOAD
  RCS_SIZE: MEDIUM
  COUNTRY_CODE: PRC
  LAUNCH_DATE: 2005-10-27
  SITE: PKMTR
  DECAY_DATE: None
  FILE: 5207233
  GP_ID: 323986649
  TLE_LINE0: 0 BEIJING 1 (TSINGHUA)
  TLE_LINE1: 1 28890U 05043A   26140

In [9]:
import json
import pandas as pd

# TLE 데이터 로드
with open("../project/01_data/raw/tle_eo_sar.json") as f:
    tle_data = json.load(f)

df_tle = pd.DataFrame(tle_data)[[
    'OBJECT_NAME', 'NORAD_CAT_ID', 'COUNTRY_CODE',
    'LAUNCH_DATE', 'PERIOD', 'INCLINATION',
    'APOAPSIS', 'PERIAPSIS', 'RCS_SIZE', 'DECAY_DATE'
]]
df_tle['NORAD_CAT_ID'] = df_tle['NORAD_CAT_ID'].astype(int)

# UCS DB 로드
df_ucs = pd.read_csv("../project/01_data/raw/ucs_eo_sar_filtered.csv")
df_ucs = df_ucs.rename(columns={'NORAD Number': 'NORAD_CAT_ID'})

# JOIN
df_combined = df_tle.merge(
    df_ucs[['NORAD_CAT_ID', 'sensor_type', 'Detailed Purpose']],
    on='NORAD_CAT_ID',
    how='left'
)

print(f"통합 위성 수: {len(df_combined)}개")
print(f"\nsensor_type 분포:")
print(df_combined['sensor_type'].value_counts())
print(f"\n샘플:")
print(df_combined.head(3))

# 저장
df_combined.to_csv("../project/01_data/raw/satellite_info.csv", index=False)
print("\n저장 완료: satellite_info.csv")

통합 위성 수: 419개

sensor_type 분포:
sensor_type
EO     367
SAR     52
Name: count, dtype: int64

샘플:
            OBJECT_NAME  NORAD_CAT_ID COUNTRY_CODE LAUNCH_DATE  PERIOD  \
0  BEIJING 1 (TSINGHUA)         28890          PRC  2005-10-27  98.393   
1             DMC 3-FM1         40715           UK  2015-07-10  96.924   
2             DMC 3-FM2         40716           UK  2015-07-10  96.923   

  INCLINATION APOAPSIS PERIAPSIS RCS_SIZE DECAY_DATE sensor_type  \
0     98.2880  692.378   671.327   MEDIUM       None          EO   
1     97.6490  615.957   606.795    LARGE       None          EO   
2     97.6546  616.159   606.574    LARGE       None          EO   

  Detailed Purpose  
0  Optical Imaging  
1  Optical Imaging  
2  Optical Imaging  

저장 완료: satellite_info.csv


In [11]:
import requests

USERNAME = "dejuakim@gmail.com"
PASSWORD = "siawevengersaiffel"

login_url = "https://www.space-track.org/ajaxauth/login"
session = requests.Session()
resp = session.post(login_url, data={"identity": USERNAME, "password": PASSWORD})
print(f"로그인 상태: {resp.status_code}")

# 로그인 확인
test_url = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp/NORAD_CAT_ID/25544/format/json"
)
test = session.get(test_url)
print(f"테스트 응답: {test.status_code}")
print(test.text[:100])

로그인 상태: 200
테스트 응답: 200
[{"CCSDS_OMM_VERS":"3.0","COMMENT":"GENERATED VIA SPACE-TRACK.ORG API","CREATION_DATE":"2026-05-20T1


In [13]:
import pandas as pd

url_new = (
    "https://www.space-track.org/basicspacedata/query"
    "/class/gp"
    "/OBJECT_TYPE/PAYLOAD"
    "/MEAN_MOTION/13.7--15.8"
    "/INCLINATION/>40"
    "/ECCENTRICITY/<0.25"
    "/LAUNCH_DATE/>2023-05-01"
    "/DECAY_DATE/null-val"
    "/EPOCH/>now-30"
    "/orderby/OBJECT_NAME"
    "/format/json"
)

response = session.get(url_new)
new_sats = response.json()
print(f"2023년 5월 이후 발사 LEO 위성: {len(new_sats)}개")

df_new = pd.DataFrame(new_sats)[['OBJECT_NAME', 'NORAD_CAT_ID', 'LAUNCH_DATE', 'COUNTRY_CODE']]
print(f"\n샘플 (상위 20개):")
print(df_new.head(20).to_string())

2023년 5월 이후 발사 LEO 위성: 8989개

샘플 (상위 20개):
      OBJECT_NAME NORAD_CAT_ID LAUNCH_DATE COUNTRY_CODE
0          3CAT-4        60236  2024-07-09          SPN
1       6GSTARLAB        66683  2025-11-28          SPN
2   A-SEANSAT-PG1        57197  2023-06-27         MALA
3    AAC-AIS-SAT2        56996  2023-06-12           UK
4    AAC-AIS-SAT3        58997  2023-11-11           UK
5    AAC-HSI-SAT2        56995  2023-06-12           UK
6    AAC-HSI-SAT3        58848  2023-11-11           UK
7         AAC-IO1        68498  2026-03-30           UK
8         AC1-001        66744  2025-11-28           UK
9         AC1-002        66745  2025-11-28           UK
10        AC1-003        66689  2025-11-28           UK
11    ACCENTURE-1        66767  2025-11-28           UK
12          ACS 3        59588  2024-04-23           US
13        ADRAS-J        58992  2024-02-18          JPN
14           AE1A        68487  2026-03-30          JPN
15           AE1C        62649  2025-01-14          JPN
16   

In [14]:
import pandas as pd

df_new = pd.DataFrame(new_sats)

# 이름 기반 EO/SAR 키워드 필터링
eo_keywords = [
    'PLANET', 'SKYSAT', 'WORLDVIEW', 'SPACEYE', 'PELICAN', 'LEGION',
    'SUPERVIEW', 'JILIN', 'GAOFEN', 'DOVE', 'FLOCK', 'SKYSAT',
    'KOMPSAT', 'PLEIADES', 'SPOT', 'SENTINEL', 'LANDSAT', 'ZHUHAI',
    'BEIJING', 'TRISAT', 'SATELLOGIC', 'NEWSAT', 'ICEYE'
]

sar_keywords = [
    'SAR', 'RADAR', 'ICEYE', 'CAPELLA', 'UMBRA', 'SYNSPECTIVE',
    'SAOCOM', 'NISAR', 'ALOS', 'SARAH', 'SICD'
]

def classify_new_sat(name):
    name_upper = name.upper()
    for kw in sar_keywords:
        if kw in name_upper:
            return 'SAR'
    for kw in eo_keywords:
        if kw in name_upper:
            return 'EO'
    return None  # 분류 불가

df_new['sensor_type'] = df_new['OBJECT_NAME'].apply(classify_new_sat)

# 분류된 것만 필터링
df_new_filtered = df_new[df_new['sensor_type'].notna()].copy()

print(f"전체: {len(df_new)}개")
print(f"EO/SAR 분류 완료: {len(df_new_filtered)}개")
print(f"\nsensor_type 분포:")
print(df_new_filtered['sensor_type'].value_counts())
print(f"\n샘플:")
print(df_new_filtered[['OBJECT_NAME', 'NORAD_CAT_ID', 'LAUNCH_DATE', 'COUNTRY_CODE', 'sensor_type']].head(20).to_string())

전체: 8989개
EO/SAR 분류 완료: 232개

sensor_type 분포:
sensor_type
EO     160
SAR     72
Name: count, dtype: int64

샘플:
             OBJECT_NAME NORAD_CAT_ID LAUNCH_DATE COUNTRY_CODE sensor_type
43                ALOS-4        60182  2024-07-01          JPN         SAR
82   CAPELLA-11 (ACADIA)        57693  2023-08-23           US         SAR
83   CAPELLA-13 (ACADIA)        60419  2024-08-11           US         SAR
84   CAPELLA-14 (ACADIA)        59444  2024-04-07           US         SAR
85   CAPELLA-15 (ACADIA)        60544  2024-08-16           US         SAR
86   CAPELLA-16 (ACADIA)        65318  2025-08-26           US         SAR
87   CAPELLA-17 (ACADIA)        64583  2025-06-23           US         SAR
88   CAPELLA-18 (ACADIA)        67385  2026-01-11           US         SAR
89   CAPELLA-19 (ACADIA)        67384  2026-01-11           US         SAR
90   CAPELLA-20 (ACADIA)        68485  2026-03-30           US         SAR
92               CARBSAR        67397  2026-01-11           UK  

In [16]:
import pandas as pd

# 기존 satellite_info.csv 로드
df_sat_info = pd.read_csv("../project/01_data/raw/satellite_info.csv")

# SpaceEye-T 포함 여부 확인
spaceye = df_new_filtered[df_new_filtered['OBJECT_NAME'].str.contains('SPACEYE', case=False, na=False)]
print(f"SpaceEye 관련 위성:")
print(spaceye[['OBJECT_NAME', 'NORAD_CAT_ID', 'LAUNCH_DATE', 'sensor_type']].to_string())

# 기존 419개와 중복 확인
existing_norads = set(df_sat_info['NORAD_CAT_ID'].astype(int).tolist())
new_norads = set(df_new_filtered['NORAD_CAT_ID'].astype(int).tolist())
overlap = existing_norads & new_norads
print(f"\n기존 419개와 중복: {len(overlap)}개")
print(f"신규 추가될 위성: {len(new_norads - existing_norads)}개")

SpaceEye 관련 위성:
Empty DataFrame
Columns: [OBJECT_NAME, NORAD_CAT_ID, LAUNCH_DATE, sensor_type]
Index: []

기존 419개와 중복: 0개
신규 추가될 위성: 232개


In [17]:
# SpaceEye-T 직접 검색
spaceye_search = df_new[df_new['OBJECT_NAME'].str.contains('SPACE', case=False, na=False)]
print(f"SPACE 포함 위성:")
print(spaceye_search[['OBJECT_NAME', 'NORAD_CAT_ID', 'LAUNCH_DATE']].to_string())

SPACE 포함 위성:
                OBJECT_NAME NORAD_CAT_ID LAUNCH_DATE
100        CENTISPACE-1 S10        62579  2025-01-13
101        CENTISPACE-1 S11        62580  2025-01-13
102        CENTISPACE-1 S12        62581  2025-01-13
103        CENTISPACE-1 S13        62582  2025-01-13
104        CENTISPACE-1 S14        62583  2025-01-13
105        CENTISPACE-1 S15        62584  2025-01-13
106        CENTISPACE-1 S16        62585  2025-01-13
107         CENTISPACE-1 S7        62576  2025-01-13
108         CENTISPACE-1 S8        62577  2025-01-13
109         CENTISPACE-1 S9        62578  2025-01-13
110         CENTISPACE-2 S1        68350  2026-03-22
111        CENTISPACE-2 S10        68359  2026-03-22
112         CENTISPACE-2 S2        68351  2026-03-22
113         CENTISPACE-2 S3        68352  2026-03-22
114         CENTISPACE-2 S4        68353  2026-03-22
115         CENTISPACE-2 S5        68354  2026-03-22
116         CENTISPACE-2 S6        68355  2026-03-22
117         CENTISPACE-2 S7      

In [18]:
# SPACEEYE 키워드 추가 후 재분류
eo_keywords = [
    'PLANET', 'SKYSAT', 'WORLDVIEW', 'SPACEEYE', 'SPACEYE', 'PELICAN', 'LEGION',
    'SUPERVIEW', 'JILIN', 'GAOFEN', 'DOVE', 'FLOCK',
    'KOMPSAT', 'PLEIADES', 'SPOT', 'SENTINEL', 'LANDSAT', 'ZHUHAI',
    'BEIJING', 'TRISAT', 'SATELLOGIC', 'NEWSAT'
]

sar_keywords = [
    'SAR', 'RADAR', 'ICEYE', 'CAPELLA', 'UMBRA', 'SYNSPECTIVE',
    'SAOCOM', 'NISAR', 'ALOS', 'SARAH', 'SICD'
]

def classify_new_sat(name):
    name_upper = name.upper()
    for kw in sar_keywords:
        if kw in name_upper:
            return 'SAR'
    for kw in eo_keywords:
        if kw in name_upper:
            return 'EO'
    return None

df_new['sensor_type'] = df_new['OBJECT_NAME'].apply(classify_new_sat)
df_new_filtered = df_new[df_new['sensor_type'].notna()].copy()

# SpaceEye-T1 확인
spaceye = df_new_filtered[df_new_filtered['OBJECT_NAME'].str.contains('SPACEEYE', case=False, na=False)]
print(f"SpaceEye 관련 위성:")
print(spaceye[['OBJECT_NAME', 'NORAD_CAT_ID', 'LAUNCH_DATE', 'sensor_type']].to_string())

print(f"\n전체 신규 EO/SAR: {len(df_new_filtered)}개")
print(df_new_filtered['sensor_type'].value_counts())

SpaceEye 관련 위성:
      OBJECT_NAME NORAD_CAT_ID LAUNCH_DATE sensor_type
1352  SPACEEYE-T1        63229  2025-03-15          EO

전체 신규 EO/SAR: 233개
sensor_type
EO     161
SAR     72
Name: count, dtype: int64


In [20]:
import json
import pandas as pd

# ── 기존 satellite_info.csv 로드 ──────────────────────────
df_existing = pd.read_csv("../project/01_data/raw/satellite_info.csv")

# ── 신규 위성 정보 구성 ───────────────────────────────────
df_new_info = df_new_filtered[[
    'OBJECT_NAME', 'NORAD_CAT_ID', 'COUNTRY_CODE',
    'LAUNCH_DATE', 'sensor_type'
]].copy()
df_new_info.columns = [
    'OBJECT_NAME', 'NORAD_CAT_ID', 'COUNTRY_CODE',
    'LAUNCH_DATE', 'sensor_type'
]

# ── 통합 ──────────────────────────────────────────────────
df_combined = pd.concat([df_existing, df_new_info], ignore_index=True)
df_combined = df_combined.drop_duplicates(subset='NORAD_CAT_ID')

print(f"기존: {len(df_existing)}개")
print(f"신규: {len(df_new_info)}개")
print(f"통합: {len(df_combined)}개")
print(f"\nsensor_type 분포:")
print(df_combined['sensor_type'].value_counts())

# ── 저장 ──────────────────────────────────────────────────
df_combined.to_csv("../project/01_data/raw/satellite_info.csv", index=False)
print("\n저장 완료: satellite_info.csv")

# ── 신규 NORAD ID 추출 (TLE 수집용) ──────────────────────
new_norad_ids = df_new_info['NORAD_CAT_ID'].astype(int).tolist()
print(f"\n신규 TLE 수집 대상: {len(new_norad_ids)}개")

기존: 419개
신규: 233개
통합: 652개

sensor_type 분포:
sensor_type
EO     528
SAR    124
Name: count, dtype: int64

저장 완료: satellite_info.csv

신규 TLE 수집 대상: 233개


In [21]:
import time
import json

# 신규 TLE 수집
def fetch_tle_batch(norad_batch):
    ids = ",".join(str(n) for n in norad_batch)
    url = (
        "https://www.space-track.org/basicspacedata/query"
        f"/class/gp/NORAD_CAT_ID/{ids}"
        "/orderby/NORAD_CAT_ID"
        "/format/json"
    )
    response = session.get(url)
    if response.status_code != 200:
        return []
    result = response.json()
    if isinstance(result, list):
        result = [r for r in result if isinstance(r, dict)]
    return result

batch_size = 100
new_tle = []

for i in range(0, len(new_norad_ids), batch_size):
    batch = new_norad_ids[i:i+batch_size]
    result = fetch_tle_batch(batch)
    new_tle.extend(result)
    print(f"수집 완료: {i+len(batch)}/{len(new_norad_ids)} | 현재: {len(new_tle)}개")
    time.sleep(2)

print(f"\n신규 TLE 수집: {len(new_tle)}개")

# 기존 TLE 로드
with open("../project/01_data/raw/tle_eo_sar.json") as f:
    existing_tle = json.load(f)

print(f"기존 TLE: {len(existing_tle)}개")

# 통합 (중복 제거)
existing_norad_set = set(str(t['NORAD_CAT_ID']) for t in existing_tle)
new_tle_dedup = [t for t in new_tle if str(t['NORAD_CAT_ID']) not in existing_norad_set]

all_tle = existing_tle + new_tle_dedup
print(f"통합 TLE: {len(all_tle)}개")

# 저장
with open("../project/01_data/raw/tle_eo_sar.json", "w") as f:
    json.dump(all_tle, f)

print("저장 완료: tle_eo_sar.json")

수집 완료: 100/233 | 현재: 100개
수집 완료: 200/233 | 현재: 200개
수집 완료: 233/233 | 현재: 233개

신규 TLE 수집: 233개
기존 TLE: 419개
통합 TLE: 652개
저장 완료: tle_eo_sar.json


In [22]:
import pandas as pd

df = pd.read_csv("../project/01_data/raw/satellite_info.csv")

print("sensor_type 분포:")
print(df['sensor_type'].value_counts())

print("\nDetailed Purpose 분포:")
print(df['Detailed Purpose'].value_counts())

print("\nEO 위성의 Detailed Purpose:")
print(df[df['sensor_type']=='EO']['Detailed Purpose'].value_counts())

print("\nSAR 위성의 Detailed Purpose:")
print(df[df['sensor_type']=='SAR']['Detailed Purpose'].value_counts())

sensor_type 분포:
sensor_type
EO     528
SAR    124
Name: count, dtype: int64

Detailed Purpose 분포:
Detailed Purpose
Optical Imaging                                          326
Radar Imaging                                             38
Multispectral Imaging                                     13
Hyperspectral Imaging                                     13
Radar Imaging (SAR)                                       12
Video Imaging                                              7
Synthetic Aperture Imaging                                 2
Optical Imaging (video)                                    2
Infrared Imaging                                           1
Subsurface Imaging                                         1
Optical Imaging/Automatic Identification System (AIS)      1
Optical/Hyperspectral Imaging                              1
Optical/Infrared Imaging                                   1
Optical Imaging/Infrared Imaging                           1
Name: count, dtype: int64

EO 위

In [23]:
import pandas as pd

df = pd.read_csv("../project/01_data/raw/satellite_info.csv")
print("컬럼 목록:")
print(df.columns.tolist())
print(f"\n샘플:")
print(df[['OBJECT_NAME', 'sensor_type', 'Detailed Purpose', 'APOAPSIS']].head(5))

컬럼 목록:
['OBJECT_NAME', 'NORAD_CAT_ID', 'COUNTRY_CODE', 'LAUNCH_DATE', 'PERIOD', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'RCS_SIZE', 'DECAY_DATE', 'sensor_type', 'Detailed Purpose']

샘플:
            OBJECT_NAME sensor_type Detailed Purpose  APOAPSIS
0  BEIJING 1 (TSINGHUA)          EO  Optical Imaging   692.378
1             DMC 3-FM1          EO  Optical Imaging   615.957
2             DMC 3-FM2          EO  Optical Imaging   616.159
3             DMC 3-FM3          EO  Optical Imaging   615.620
4          PATHFINDER 1          EO  Optical Imaging   687.738


In [24]:
import pandas as pd
df = pd.read_csv("../project/01_data/raw/satellite_info.csv")
print(df['RCS_SIZE'].value_counts())
print()
print(df.groupby(['RCS_SIZE', 'sensor_type']).size())

RCS_SIZE
MEDIUM    224
SMALL     134
LARGE      61
Name: count, dtype: int64

RCS_SIZE  sensor_type
LARGE     EO              36
          SAR             25
MEDIUM    EO             197
          SAR             27
SMALL     EO             134
dtype: int64


In [25]:
import pandas as pd
df = pd.read_csv("../project/01_data/processed/satellite_passes.csv")
print(f"전체 근접 통과 기록: {len(df)}건")
print(f"관련 위성 수: {df['satellite_name'].nunique()}개")
print(f"관련 이벤트 수: {df['SQLDATE'].nunique()}개")

전체 근접 통과 기록: 1451건
관련 위성 수: 388개
관련 이벤트 수: 224개


In [1]:
import pandas as pd
df = pd.read_csv("../project/01_data/raw/satellite_info.csv")
print(df['RCS_SIZE'].value_counts())

RCS_SIZE
MEDIUM    224
SMALL     134
LARGE      61
Name: count, dtype: int64


In [2]:
import pandas as pd
df = pd.read_csv("../project/01_data/raw/satellite_info.csv")
small_sats = df[df['RCS_SIZE'] == 'SMALL'][['OBJECT_NAME', 'sensor_type', 'LAUNCH_DATE', 'COUNTRY_CODE']]
print(f"SMALL 위성 수: {len(small_sats)}")
print(small_sats.to_string())

SMALL 위성 수: 134
            OBJECT_NAME sensor_type LAUNCH_DATE COUNTRY_CODE
5            FLOCK 3P 4          EO  2017-02-15           US
6            FLOCK 3P 2          EO  2017-02-15           US
7            FLOCK 3P 3          EO  2017-02-15           US
8           FLOCK 3P 12          EO  2017-02-15           US
9           FLOCK 3P 11          EO  2017-02-15           US
10          FLOCK 3P 58          EO  2017-02-15           US
11          FLOCK 3P 57          EO  2017-02-15           US
12          FLOCK 3P 34          EO  2017-02-15           US
13          FLOCK 3P 33          EO  2017-02-15           US
14          FLOCK 3P 49          EO  2017-02-15           US
15          FLOCK 3P 61          EO  2017-02-15           US
16          FLOCK 3P 54          EO  2017-02-15           US
17          FLOCK 3P 23          EO  2017-02-15           US
18          FLOCK 3P 76          EO  2017-02-15           US
19          FLOCK 3P 32          EO  2017-02-15           US
20      